#### מגישות :
#### Ayelet Marks 207266966 איילת מרקס
#### Sofi Molochny 213215841 סופי מולוצ'ני 

# Imports and set up

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time 
import json
import os


## Before Executing:
!!! <b>YOU MUST PROVIDE AN API KEY IN THE FORM OF %env API_KEY PASTE_YOUR_KEY_HERE</b>

In [10]:
# %env API_KEY PASTE_YOUR_KEY_HERE

In [5]:
os.getenv("API_KEY") is not None

False

# Constants

נמפה את השמות בעברית ובאנגלית ונתאים את הטיפוס המתאים : 

In [7]:
CSV_FILE_NAME = "scrape_results.csv"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36'
}

FIELD_MAPPINGS = {
    "פרטי הנכס": ("property_type", str),
    "שכונה": ("neighborhood", str),
    "כתובת": ("address", str),
    "חדרים": ("room_num", float),
    "קומה": ("floor", int),
    "שטח בנוי": ("area", int),
    "שטח גינה": ("garden_area", int),
    "מספר ימים לכניסה": ("days_to_enter", int),
    "תשלומים בשנה": ("num_of_payments", int),
    "ארנונה חודשית": ("monthly_arnona", int),
    "ועד בית בחודש": ("building_tax", int),
    "קומות בבניין": ("total_floors", int),
    "תיאור": ("description", str),
    "חניה": ("has_parking", lambda x: 1 if x == "יש" else 0),
    "מחסן": ("has_storage", lambda x: 1 if x == "יש" else 0),
    "מעלית": ("elevator", lambda x: 1 if x == "יש" else 0),
    "מזגן": ("ac", lambda x: 1 if x == "יש" else 0),
    "נגישות": ("handicap", lambda x: 1 if x == "יש" else 0),
    "סורגים": ("has_bars", lambda x: 1 if x == "יש" else 0),
    "ממד": ('has_safe_room', lambda x: 1 if x == "יש" else 0),
    "מרפסת": ("has_balcony", lambda x: 1 if x == "יש" else 0),
    "מרוהטת": ("is_furnished", lambda x: 1 if x == "כן" else 0),
    "משופצת": ("is_renovated", lambda x: 1 if x == "כן" else 0),
    "מחיר": ("price", lambda x: float(x.replace(",", "")) if x else None),
    "מספר תמונות מצורפות": ("num_of_images", int),
    "מרחק ממרכז העיר במטרים":("distance_from_center" , float),
}

BINARY_FEATURES = {
    "חניה": "has_parking",
    "מחסן": "has_storage",
    "מעלית": "elevator",
    "מזגן": "ac",
    "נגישות": "handicap",
    "סורגים": "has_bars",
    "ממד": "has_safe_room",
    "מרפסת": "has_balcony",
    "מרוהטת": "is_furnished",
    "משופצת": "is_renovated"
}

GOOGLE_API_KEY = os.getenv("API_KEY").strip('"').strip("'")

AttributeError: 'NoneType' object has no attribute 'strip'

# Test URL is valid

#### שליפת מידע מהצפון הישן האזור הצפוני 

In [9]:
url = "https://www.ad.co.il/nadlanrent?sp276=17414&sp277=17463"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36'} # This is chrome, you can set whatever browser you like

response = requests.get(url,headers= headers)
if response.status_code == 200:
    print("Success")
    result_page = BeautifulSoup(response.content, 'html.parser')
else:
    print("Failure")

Success


#### שליפת מידע מהמע"ר הצפוני 

In [15]:
url = "https://www.ad.co.il/nadlanrent?sp275=17413&sp277=37838"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36'} # This is chrome, you can set whatever browser you like

response = requests.get(url,headers= headers)
if response.status_code == 200:
    print("Success")
    result_page = BeautifulSoup(response.content, 'html.parser')
else:
    print("Failure")

Success


# Functions definitions

In [16]:
def get_distance_from_center(address: str) -> float:
    """Functin returns the distance of an adress from כיכר דיזינגוף, תל אביב. All using the Google Masps API.
    Make sure you are using a valid API key!

    Args:
        address (str): the adress in question.

    Returns:
        float: distance in meters
    """
    url = "https://routes.googleapis.com/directions/v2:computeRoutes"
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': GOOGLE_API_KEY,
        'X-Goog-FieldMask': 'routes.distanceMeters'
    }
    body = {
        "origin": {"address": address + ", תל אביב"},
        "destination": {"address": "כיכר דיזינגוף, תל אביב"},
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_AWARE"
    }
    try:
        response = requests.post(url, headers=headers, data=json.dumps(body))
        if response.status_code == 200:
            data = response.json()
            distance_meters = data['routes'][0]['distanceMeters']
            return float(distance_meters)
        else:
            print(f"שגיאה ({response.status_code}) במציאת מרחק לכתובת: {address}")
            print(response.text)
            return None
    except Exception as e:
        print(f"שגיאה בשאילת API: {e}")
        return None


def get_apartment_info(apartment_link: str) -> dict:
    """Function extracts relevant info from an apartment link.

    Args:
        apartment_link (str): The URL to an apartment ad.

    Returns:
        dict: A dictionary represantation of the wanted scraped fields.
    """
    apartment_dict = {v[0]: 0 if v[1] in [int, float] else '' for k, v in FIELD_MAPPINGS.items()}
    apartment_dict['link'] = apartment_link
    apartment_dict["distance_from_center"] = 0.0  # נחשב אחר כך

    try:
        response = requests.get(apartment_link, headers=HEADERS)
        if response.status_code != 200:
            return apartment_dict

        soup = BeautifulSoup(response.content, 'html.parser')

        table = soup.find('table', {'class': 'table table-sm mb-4'})
        if table:
            rows = table.find_all("tr")
            for row in rows:
                cells = row.find_all("td")
                if len(cells) == 2:
                    hebrew_key = cells[0].text.strip()
                    value = cells[1].text.strip()
                    if hebrew_key == "קומה":
                        value = value.split()[0].strip()
                        if value == "קרקע":
                            value = 0
                        try:
                            apartment_dict["total_floors"] = int(cells[1].text.strip().split()[-1].strip())
                        except:
                            apartment_dict["total_floors"] = 0
                    if hebrew_key in FIELD_MAPPINGS:
                        english_key, data_type = FIELD_MAPPINGS[hebrew_key]
                        try:
                            apartment_dict[english_key] = data_type(value)
                        except ValueError:
                            apartment_dict[english_key] = 0 if data_type in [int, float] else ''

        description_p = soup.select_one('div.p-3 p.text-word-break')
        if description_p:
            apartment_dict["description"] = " ".join(description_p.get_text(strip=True).split())

        price_div = soup.find('div', class_='d-flex justify-content-between')
        if price_div:
            h2_tags = price_div.find_all('h2', class_='card-title')
            if len(h2_tags) >= 2:
                price_text = h2_tags[1].get_text(strip=True).replace(",", "").replace("₪", "")
                try:
                    apartment_dict["price"] = float(price_text)
                except ValueError:
                    apartment_dict["price"] = 0

        for key in BINARY_FEATURES.values():
            apartment_dict[key] = 0

        features_container = soup.find('div', class_='card-icons flex-wrap d-flex h-100')
        if features_container:
            feature_divs = features_container.find_all('div', class_='card-icon')
            for feature_div in feature_divs:
                is_disabled = 'disabled' in feature_div.get('class', [])
                span_tag = feature_div.find('span', class_='px-1')
                if span_tag:
                    hebrew_label = span_tag.text.strip()
                    if hebrew_label in BINARY_FEATURES:
                        apartment_dict[BINARY_FEATURES[hebrew_label]] = 0 if is_disabled else 1

        images_container = soup.find('div', class_='col-12 d-flex mt-3 justify-content-center flex-wrap')
        if images_container:
            images = images_container.find_all('img')
            apartment_dict['num_of_images'] = len(images)

        return apartment_dict

    except Exception as e:
        print(f"שגיאה בקישור {apartment_link}: {e}")
        return apartment_dict



def get_apartment_links(neighborhood_url: str) -> list:
    """Given a neighborhood URL, the function extracts the individiual apartment ad links.

    Args:
        neighborhood_url (str): URL of a neightborhood search.

    Returns:
        list: List of urls of apartments.
    """
    all_links = set()
    page = 1
    while True:
        url = f"{neighborhood_url}&page={page}"
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            print(f"שגיאה בטעינת עמוד {page}")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        cards = soup.find_all('div', class_='card-body p-md-3')
        if not cards:
            break

        page_links = set()
        for card in cards:
            a_tag = card.find('a', href=True)
            if a_tag:
                h2 = a_tag.find('h2', class_='card-title mb-0 mb-sm-1')
                if h2:
                    href = a_tag['href']
                    full_link = "https://www.ad.co.il" + href
                    page_links.add(full_link)

        if not page_links - all_links:
            print("לא נמצאו קישורים חדשים, עצירה.")
            break

        all_links.update(page_links)
        print(f"עמוד {page}: נוספו {len(page_links)} קישורים חדשים")
        page += 1
        time.sleep(1)

    return list(all_links)


def scrape_neighborhood_to_csv(neighborhood_url: str, csv_filename: str = CSV_FILE_NAME) -> pd.DataFrame:
    """Fnciton creates a dataframe of the apartments in the neighborhood, and saves it in CSV.

    Args:
        neighborhood_url (str): URL of the neighborood.
        csv_filename (str, optional): Path to save the csv file. Defaults to CSV_FILE_NAME.

    Returns:
        pd.DataFrame: DataFrame of the saves CSV file.
    """
    print("מתחילה לאסוף קישורים מהשכונה...")
    links = get_apartment_links(neighborhood_url)
    print(f"נמצאו {len(links)} קישורים לדירות")
    all_apartments = []
    for i, link in enumerate(links):
        print(f"({i + 1}/{len(links)}) אוסף מידע מהמודעה: {link}")
        data = get_apartment_info(link)
        all_apartments.append(data)
        time.sleep(1)

    df = pd.DataFrame(all_apartments)
    file_exists = os.path.exists(csv_filename)
    # df.to_csv(csv_filename, index=False, encoding='utf-8-sig', mode='a', header=not file_exists)
    mask = df["distance_from_center"].isnull() | (df["distance_from_center"] == 0)
    for idx in df[mask].index:
        address = df.at[idx, "address"]
        if address:
            distance = get_distance_from_center(address)
            df.at[idx, "distance_from_center"] = distance if distance is not None else 0.0

    df.to_csv(csv_filename, index=False, encoding='utf-8-sig', mode='a', header=not file_exists)
    print(f" המרחקים עודכנו והקובץ נשמר: {csv_filename}")
    return df



# Scraping

In [18]:
csv_filename_of_run = "apartment_data.csv"
print("Saving file to", csv_filename_of_run)

Saving file to scraping_results_2025-04-13T20-22-38.csv


In [19]:
neighborhood_url_maar = "https://www.ad.co.il/nadlanrent?sp275=17413&sp276=17414&sp277=37838"
df_maar = scrape_neighborhood_to_csv(neighborhood_url_maar, csv_filename_of_run)
neighborhood_url_zafon = "https://www.ad.co.il/nadlanrent?sp275=17413&sp277=17463"
df_zafon_yashan = scrape_neighborhood_to_csv(neighborhood_url_zafon, csv_filename_of_run)

מתחילה לאסוף קישורים מהשכונה...
עמוד 1: נוספו 13 קישורים חדשים
לא נמצאו קישורים חדשים, עצירה.
נמצאו 13 קישורים לדירות
(1/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/16191257
(2/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13527391
(3/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13055776
(4/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13074083
(5/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13095815
(6/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/16191149
(7/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/14300856
(8/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13335935
(9/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13090263
(10/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13519295
(11/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/16183419
(12/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/13621278
(13/13) אוסף מידע מהמודעה: https://www.ad.co.il/ad/16191246
✅ המרחקים עודכנו והקובץ נשמר: scraping_results_2025-04-13T20-22-38.csv
מתחילה לאסוף קישורים מהשכונה...
עמוד 1: 